In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

train_scored = pd.read_csv("../data/processed/train_scored.csv")
validation_scored = pd.read_csv("../data/processed/validation_scored.csv")
test_scored = pd.read_csv("../data/processed/test_scored.csv")

target = "SeriousDlqin2yrs"

final_variables = [
    "RevolvingUtilizationOfUnsecuredLines",
    "age",
    "MonthlyIncome",
    "DebtRatio",
    "NumberOfOpenCreditLinesAndLoans",
    "NumberRealEstateLoansOrLines",
    "NumberOfDependents",
    "NumberOfTimes90DaysLate"
]

# Validation & Monitoring

The purpose of validation and monitoring is to assess whether the scorecard remains stable and reliable after development.

Key areas reviewed:

- Out-of-time validation
- Population Stability Index
- Characteristic Stability Index
- Calibration monitoring
- Reject inference considerations

## Out-of-Time Validation

A true out-of-time validation requires a time-based holdout sample, for example applications from a later calendar period.

The current dataset does not contain an application date or observation date. Therefore, a true out-of-time validation cannot be performed.

Instead, the test sample is used as an unseen holdout sample for model validation.

## PSI — Population Stability Index

In [ ]:
def calculate_psi(expected, actual, buckets=10):
    expected = pd.Series(expected).dropna()
    actual = pd.Series(actual).dropna()

    breakpoints = np.percentile(
        expected,
        np.linspace(0, 100, buckets + 1)
    )

    breakpoints = np.unique(breakpoints)

    expected_bins = pd.cut(
        expected,
        bins=breakpoints,
        include_lowest=True
    )

    actual_bins = pd.cut(
        actual,
        bins=breakpoints,
        include_lowest=True
    )

    expected_pct = (
        expected_bins
        .value_counts(normalize=True)
        .sort_index()
    )

    actual_pct = (
        actual_bins
        .value_counts(normalize=True)
        .sort_index()
    )

    psi_table = pd.DataFrame({
        "expected_pct": expected_pct,
        "actual_pct": actual_pct
    })

    psi_table = psi_table.fillna(0.0001)

    psi_table["psi"] = (
        (psi_table["actual_pct"] - psi_table["expected_pct"])
        *
        np.log(psi_table["actual_pct"] / psi_table["expected_pct"])
    )

    return psi_table, psi_table["psi"].sum()

In [ ]:
psi_validation_table, psi_validation = calculate_psi(
    train_scored["score"],
    validation_scored["score"]
)

psi_validation

In [ ]:
psi_validation_table

## PSI: Train vs Test

In [ ]:
psi_test_table, psi_test = calculate_psi(
    train_scored["score"],
    test_scored["score"]
)

psi_test

In [ ]:
psi_test_table

PSI summary

In [ ]:
psi_summary = pd.DataFrame({
    "comparison": [
        "Train vs Validation",
        "Train vs Test"
    ],
    "PSI": [
        psi_validation,
        psi_test
    ]
})

psi_summary

## Population Stability Index

Population Stability Index was calculated by comparing the score distributions of the development sample against the validation and test samples.

Results:

- Train vs Validation PSI = 0.0003
- Train vs Test PSI = 0.0003

Both PSI values are substantially below the commonly accepted threshold of 0.10.

This indicates that the score distributions are highly stable and no material population shift is observed between the development, validation and test samples.

## CSI (Characteristic Stability Index)

In [ ]:
def calculate_csi(expected, actual, buckets=10):

    expected = pd.Series(expected).dropna()
    actual = pd.Series(actual).dropna()

    breakpoints = np.percentile(
        expected,
        np.linspace(0, 100, buckets + 1)
    )

    breakpoints = np.unique(breakpoints)

    expected_bins = pd.cut(
        expected,
        bins=breakpoints,
        include_lowest=True
    )

    actual_bins = pd.cut(
        actual,
        bins=breakpoints,
        include_lowest=True
    )

    expected_pct = (
        expected_bins
        .value_counts(normalize=True)
        .sort_index()
    )

    actual_pct = (
        actual_bins
        .value_counts(normalize=True)
        .sort_index()
    )

    csi_table = pd.DataFrame({
        "expected_pct": expected_pct,
        "actual_pct": actual_pct
    })

    csi_table = csi_table.fillna(0.0001)

    csi_table["csi"] = (
        (csi_table["actual_pct"] - csi_table["expected_pct"])
        *
        np.log(
            csi_table["actual_pct"]
            /
            csi_table["expected_pct"]
        )
    )

    return csi_table["csi"].sum()

In [ ]:
csi_results = []

for variable in final_variables:

    csi = calculate_csi(
        train_scored[variable],
        validation_scored[variable]
    )

    csi_results.append({
        "variable": variable,
        "CSI": csi
    })

csi_summary = (
    pd.DataFrame(csi_results)
    .sort_values(
        by="CSI",
        ascending=False
    )
)

csi_summary

## Characteristic Stability Index

CSI was calculated for all final model predictors by comparing the training sample against the validation sample.

All CSI values are substantially below 0.10, indicating that no material shift is observed in any model characteristic.

The most shifted variable was RevolvingUtilizationOfUnsecuredLines with CSI = 0.0010, which is still negligible.

Overall, the model characteristics are highly stable across samples.

## Calibration Monitoring

In [ ]:
def calibration_table(df, dataset_name):
    df = df.copy()

    df["pd_band"] = pd.qcut(
        df["pd"],
        q=10,
        labels=False,
        duplicates="drop"
    )

    result = (
        df
        .groupby("pd_band")
        .agg(
            customers=("pd", "count"),
            avg_pd=("pd", "mean"),
            observed_bad_rate=(target, "mean")
        )
        .reset_index()
    )

    result["dataset"] = dataset_name

    return result

In [ ]:
train_calibration = calibration_table(train_scored, "Train")
validation_calibration = calibration_table(validation_scored, "Validation")
test_calibration = calibration_table(test_scored, "Test")

calibration_summary = pd.concat(
    [train_calibration, validation_calibration, test_calibration],
    ignore_index=True
)

calibration_summary

## Calibration Monitoring

Calibration monitoring compares predicted PD against observed bad rate.

A well-calibrated model should produce average predicted PD values that are reasonably close to the observed default/bad rate within each risk band.

Material differences between predicted PD and observed bad rate may indicate model calibration drift and may require recalibration.